# Agente mínimo — el bucle razonar → actuar → observar en ~60 líneas

Ejemplo de testeo para la **Sesión 12**: un agente escrito a mano sobre la **Responses API** de OpenAI
(la misma que usa el proyecto del máster), con `gpt-4o-mini` para que cueste céntimos.

Qué vas a ver:
1. Una **tool** declarada con schema plano + `strict: true` (el contrato).
2. El **bucle manual**: el modelo emite la intención, *nuestro* código ejecuta, devolvemos la observación.
3. La **traza STEP N**: el modelo decide cuántas búsquedas hacer — nadie se lo programó.

Requisito: la variable de entorno `OPENAI_API_KEY` (en Colab: icono de la llave → añade el secreto y ejecuta la celda 2).

In [ ]:
# Auto-instala el SDK si este kernel no lo tiene (vale para VS Code y Colab):
import importlib.util, subprocess, sys
if importlib.util.find_spec("openai") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
    importlib.invalidate_caches()

import json
import os
from openai import OpenAI

# En Colab, lee el secreto; en local usa la variable de entorno.
try:
    from google.colab import userdata  # type: ignore
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass

assert os.environ.get("OPENAI_API_KEY"), "Falta OPENAI_API_KEY en el entorno (o en los secretos de Colab)"

client = OpenAI()  # lee OPENAI_API_KEY del entorno
MODEL = "gpt-4o-mini"
print("cliente listo ✔")

## 1. La tool: una búsqueda de tareas históricas (de juguete)

Mini-versión del `search_budgets` del proyecto: un "corpus" de 6 tareas en un diccionario.
Fíjate en dos cosas:
- El **schema es plano** (Responses API) y `strict: true`: todo en `required`, lo opcional se modela como nullable.
- La **description está escrita PARA el modelo** — es lo único que lee para decidir cuándo y cómo llamarla.

In [ ]:
CORPUS = [
    {"task": "login with OAuth (Google + GitHub)", "hours": 24},
    {"task": "user registration with email verification", "hours": 16},
    {"task": "admin dashboard with usage charts", "hours": 60},
    {"task": "KPI dashboard with export to PDF", "hours": 80},
    {"task": "Stripe payment integration", "hours": 40},
    {"task": "REST API for a product catalog", "hours": 50},
]

def search_tasks(query: str) -> list[dict]:
    """Busqueda de juguete: coincidencia de palabras (en el master es pgvector)."""
    words = set(query.lower().split())
    hits = [t for t in CORPUS if words & set(t["task"].lower().split())]
    return hits or [{"task": "NO MATCH", "hours": None}]

SEARCH_TOOL = {
    "type": "function",           # PLANO: name/description/parameters al mismo nivel
    "name": "search_tasks",
    "description": (
        "Search historical tasks analogous to ONE task you are trying to estimate. "
        "Call it once per task, with a short English query. "
        "If it returns NO MATCH, retry once with different wording."
    ),
    "parameters": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "short English search query for ONE task"}},
        "required": ["query"],
        "additionalProperties": False,
    },
    "strict": True,
}
print("tool declarada ✔")

## 2. El bucle agéntico, a mano

El corazón de la sesión. Léelo de arriba a abajo:
- el modelo **decide** (emite `function_call`s — puede pedir VARIAS en un turno),
- nuestro código **ejecuta** y devuelve cada resultado atado a su `call_id`,
- se encadena con `previous_response_id` (el servidor recuerda la conversación),
- para cuando el modelo no pide más tools — con `MAX_ITERATIONS` de red de seguridad.

In [ ]:
SYSTEM = (
    "You are an estimation agent. Break the user's project into tasks, "
    "use search_tasks to find one historical analog per task, and then answer in Spanish "
    "with a table of tasks, their analog and hours, plus the total. "
    "If a task has no analog, say so honestly instead of inventing hours."
)

MAX_ITERATIONS = 8

def run_agent(brief: str) -> str:
    response = client.responses.create(
        model=MODEL, instructions=SYSTEM, input=brief, tools=[SEARCH_TOOL], store=True
    )
    step = 0
    for _ in range(MAX_ITERATIONS):
        calls = [item for item in response.output if item.type == "function_call"]  # TODAS, no solo la 1a
        if not calls:
            break                                   # parada natural: turno sin tools
        outputs = []
        for call in calls:
            step += 1
            args = json.loads(call.arguments)
            result = search_tasks(**args)           # ← NUESTRO codigo ejecuta, no el modelo
            print(f"STEP {step}\n  action:      {call.name}({args})\n  observation: {result}\n")
            outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,            # el MISMO id que envio el modelo
                "output": json.dumps(result),       # string, no dict
            })
        response = client.responses.create(
            model=MODEL, previous_response_id=response.id,  # encadena estado en el servidor
            input=outputs, tools=[SEARCH_TOOL], store=True,
        )
    return response.output_text

print("bucle definido ✔")

## 3. A correr — y a LEER la traza

Lo importante no es la respuesta final: es la traza. ¿Cuántas búsquedas hizo? ¿Con qué queries?
Cambia el brief (añade o quita piezas) y vuelve a ejecutar: **el número de STEPs cambia solo**.
Eso —el modelo decidiendo el control de flujo— es lo que lo hace agéntico.

In [ ]:
brief = (
    "Quiero una web para mi gimnasio: registro de socios con verificación por email, "
    "pagos de cuotas con Stripe y un dashboard de administración con gráficas de uso."
)

respuesta = run_agent(brief)
print("=" * 60)
print(respuesta)

## 4. Para jugar (ejercicios de 2 minutos)

1. **Rompe la description**: cámbiala por `"Search tasks."` a secas y re-ejecuta. ¿Sigue haciendo una búsqueda por tarea, o mete todo en una query? (Es el ejercicio 2.3 de la sesión: la description dirige la decisión.)
2. **Pide algo sin análogo** ("una app de realidad virtual"): ¿reformula y reintenta? ¿Admite que no hay dato?
3. **Baja `MAX_ITERATIONS` a 2**: mira cómo el corte de seguridad le impide terminar — la diferencia entre parada natural y salvaguarda.

En el proyecto real esto mismo es `app/generation/agentic/agent_loop.py`, con pgvector
de corpus, consenso determinista para las horas y trazas tipadas (`AgentTrace`).